# Imports

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Tuple

# Hungarian Algo

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Tuple


class Hungarian:
    def __init__(self):
        self.L = []
        self.R = []
        self.C = None
        self.n = 0
        self.lm = {}
        self.rm = {}

    def load(self, file: str):
        try:
            df = pd.read_csv(file, header=None)
            print(f"Loaded {len(df.columns)} edges")

            edges = []
            for i in range(len(df.columns)):
                u = int(df.iloc[0, i])
                v = int(df.iloc[1, i])
                w = float(df.iloc[2, i])
                edges.append((u, v, w))

            lset = set(u for u, _, _ in edges)
            rset = set(v for _, v, _ in edges)

            self.L = sorted(list(lset))
            self.R = sorted(list(rset))

            print(f"L: {self.L}")
            print(f"R: {self.R}")

            self.n = max(len(self.L), len(self.R))
            self.C = np.full((self.n, self.n), float('inf'))

            self.lm = {v: i for i, v in enumerate(self.L)}
            self.rm = {v: i for i, v in enumerate(self.R)}

            for u, v, w in edges:
                i = self.lm[u]
                j = self.rm[v]
                self.C[i, j] = w

            print(f"Graph: {len(self.L)} L, {len(self.R)} R")
            return True

        except Exception as e:
            print(f"Error: {e}")
            return False

    def run(self):
        n = self.n
        c = self.C.copy()
        u = np.zeros(n)
        v = np.zeros(n)

        for j in range(n):
            if j < len(self.R):
                mv = np.min(c[:, j])
                if mv != float('inf'):
                    v[j] = mv

        mL = -np.ones(n, dtype=int)
        mR = -np.ones(n, dtype=int)

        cnt = 0

        print("\nHUNGARIAN\n")

        for s in range(len(self.L)):
            cnt += 1
            print(f"\nITERATION {cnt}")
            print(f"Proc {self.L[s]}")

            sl = np.full(n, float('inf'))
            slL = -np.ones(n, dtype=int)
            p = -np.ones(n, dtype=int)
            vis = np.zeros(n, dtype=bool)
            cr = -1
            p[s] = -2

            print("Dual start:")
            self.dual(u, v)

            for j in range(n):
                if j < len(self.R) and c[s, j] != float('inf'):
                    sl[j] = c[s, j] - u[s] - v[j]
                    slL[j] = s

            found = False

            while True:
                ms = float('inf')
                cr = -1

                for j in range(n):
                    if j < len(self.R) and not vis[j] and sl[j] < ms:
                        ms = sl[j]
                        cr = j

                if cr == -1 or ms == float('inf'):
                    break

                if ms > 1e-10:
                    for i in range(n):
                        if i < len(self.L) and p[i] != -1:
                            u[i] += ms
                    for j in range(n):
                        if j < len(self.R) and vis[j]:
                            v[j] -= ms
                    for j in range(n):
                        if j < len(self.R) and not vis[j]:
                            sl[j] -= ms

                vis[cr] = True

                if mR[cr] == -1:
                    found = True
                    break

                ml = mR[cr]
                if ml != -1:
                    p[ml] = cr

                    for j in range(n):
                        if j < len(self.R) and not vis[j] and c[ml, j] != float('inf'):
                            ns = c[ml, j] - u[ml] - v[j]
                            if ns < sl[j]:
                                sl[j] = ns
                                slL[j] = ml

            if found and cr != -1:
                tr = cr
                while tr != -1:
                    pl = slL[tr]
                    if pl != -1:
                        nr = p[pl] if p[pl] >= 0 else -1
                        mR[tr] = pl
                        mL[pl] = tr
                        tr = nr
                    else:
                        break

            print("Dual after:")
            self.dual(u, v)

            ct = 0
            for i in range(len(self.L)):
                if mL[i] != -1:
                    j = mL[i]
                    if j < len(self.R) and c[i, j] != float('inf'):
                        ct += 1
            print(f"Pairs: {ct}/{len(self.L)}")

        self.mfinal(mL, mR, u, v, c)
        return mL, mR, u, v

    def dual(self, u: np.ndarray, v: np.ndarray):
        print("u: ", end="")
        for i, val in enumerate(u):
            if i < len(self.L):
                print(f"{self.L[i]}:{val:.1f} ", end="")
        print()

        print("v: ", end="")
        for j, val in enumerate(v):
            if j < len(self.R):
                print(f"{self.R[j]}:{val:.1f} ", end="")
        print()

    def mfinal(self, mL: np.ndarray, mR: np.ndarray,
                    u: np.ndarray, v: np.ndarray, cost: np.ndarray):
        print("\nFINAL\n")

        print("Final Dual:")
        self.dual(u, v)

        print("\nMatching:")
        tot = 0
        ct = 0

        for i in range(len(self.L)):
            j = mL[i]
            if j != -1 and j < len(self.R) and cost[i, j] != float('inf'):
                ct += 1
                w = cost[i, j]
                tot += w
                print(f"  {self.L[i]} - {self.R[j]} : {w:.2f}")

        print(f"\nCost: {tot:.2f}")
        print(f"Pairs: {ct}")



# main Function call
// Can update the file name run the testcase

In [ ]:
def demo():
    algo = Hungarian()
    if algo.load('Testcase4.csv'):
        mL, mR, u, v = algo.run()

if __name__ == "__main__":
    demo()

Loaded 7 edges
L: [0, 1, 2]
R: [3, 4, 5]
Graph: 3 L, 3 R

HUNGARIAN


ITERATION 1
Proc 0
Dual start:
u: 0:0.0 1:0.0 2:0.0 
v: 3:1.0 4:2.0 5:1.0 
Dual after:
u: 0:0.0 1:0.0 2:0.0 
v: 3:1.0 4:2.0 5:1.0 
Pairs: 1/3

ITERATION 2
Proc 1
Dual start:
u: 0:0.0 1:0.0 2:0.0 
v: 3:1.0 4:2.0 5:1.0 
Dual after:
u: 0:0.0 1:0.0 2:0.0 
v: 3:1.0 4:2.0 5:1.0 
Pairs: 2/3

ITERATION 3
Proc 2
Dual start:
u: 0:0.0 1:0.0 2:0.0 
v: 3:1.0 4:2.0 5:1.0 
Dual after:
u: 0:0.0 1:0.0 2:2.0 
v: 3:1.0 4:2.0 5:1.0 
Pairs: 3/3

FINAL

Final Dual:
u: 0:0.0 1:0.0 2:2.0 
v: 3:1.0 4:2.0 5:1.0 

Matching:
  0 - 3 : 1.00
  1 - 4 : 2.00
  2 - 5 : 3.00

Cost: 6.00
Pairs: 3
